# Licență — toate evaluările GPU (un singur notebook)

Rulează pe **Kaggle GPU T4 × 2** tot ce nu poate rula pe laptop:

1. **Eval multi-model** pe setul de aur (98 cazuri): sweep de cuantizare (3B Q4/Q8/FP16) + scară (3B → 7B → 14B) + RoLlama3-8B + mini-sweep de temperatură (3B-Q8).
2. **LLM-judge faithfulness** — un model mare judecă fidelitatea fiecărui răspuns față de surse (metrică reală, nu proxy lexical).
3. **Analize de mecanism** (baseline 3B-Q4, temp 0): R7 recovery, ablație contracte ON/OFF, recuperare refuz spurios, distribuția scorurilor pentru calibrarea R7.
4. **Bundle** — arhivează toate rezultatele în `/kaggle/working/results.zip` de descărcat.

**Înainte de Run All:** Accelerator = **GPU T4 ×2**, Internet = **On**, Dataset = `licenta` (glodcosmin), bumpat la versiunea nouă.
Sweep-ul complet durează ~1–2h. Folosește **Save & Run All (Commit)** pentru kernel proaspăt.


## 1. Mediu + Ollama (2 instanțe, una per GPU)

In [ ]:
!pip uninstall -y torchvision torchaudio >/dev/null 2>&1; sudo apt-get install -y zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!nvidia-smi

In [ ]:
import subprocess, time, os, urllib.request
subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True); time.sleep(3)
HOSTS = []
for gpu_id, port in [(0, 11434), (1, 11435)]:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['OLLAMA_HOST'] = f'127.0.0.1:{port}'
    env['OLLAMA_NUM_PARALLEL'] = '2'
    env['OLLAMA_MAX_LOADED_MODELS'] = '1'
    log = open(f'/tmp/ollama_gpu{gpu_id}.log', 'w')
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=log, stderr=log, cwd='/root')
    HOSTS.append(f'http://127.0.0.1:{port}')
for host in HOSTS:
    for i in range(60):
        try:
            urllib.request.urlopen(f'{host}/api/version', timeout=2); print(f'{host}: UP'); break
        except Exception: time.sleep(1)
    else:
        !tail -30 /tmp/ollama_gpu*.log
        raise RuntimeError(f'{host} down')

## 2. Cod + date + dependențe

In [ ]:
import shutil, sys, os
from pathlib import Path

# Auto-discover dataset root. Kaggle mountează la /kaggle/input/<slug> (slug variază),
# deci căutăm un fișier-marker în loc de cale fixă.
INPUT = Path('/kaggle/input')
def _find_root():
    hits = list(INPUT.glob('**/src/licenta/generator.py'))
    if hits: return hits[0].parents[2]                 # <root>/src/licenta/generator.py -> <root>
    hits = list(INPUT.glob('**/cezicelegea_dump.json'))
    if hits: return hits[0].parent.parent if hits[0].parent.name == 'data' else hits[0].parent
    return None
SRC = _find_root()
if SRC is None:
    print('Dataset negăsit. Conținut /kaggle/input (folosește Add Input ca să atașezi dataset-ul):')
    for q in sorted(INPUT.glob('*')): print('  ', q)
    raise FileNotFoundError('Nu găsesc src/licenta/generator.py. Verifică dataset-ul atașat.')
print('Dataset root:', SRC)

WORK = Path('/kaggle/working/licenta')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
shutil.copytree(SRC / 'src', WORK / 'src')
if (SRC / 'scripts').exists():
    shutil.copytree(SRC / 'scripts', WORK / 'scripts')   # analizele de mecanism (ablation/refusal/retrieval_scores)
(WORK / 'data' / 'eval' / 'runs').mkdir(parents=True)
shutil.copy(SRC / 'data' / 'cezicelegea_dump.json', WORK / 'data' / 'cezicelegea_dump.json')
shutil.copy(SRC / 'data' / 'eval' / 'gold_set.json', WORK / 'data' / 'eval' / 'gold_set.json')
sys.path.insert(0, str(WORK / 'src')); os.chdir(WORK)
print('Working dir:', os.getcwd())

In [ ]:
!pip install -q 'chromadb>=1.5' 'sentence-transformers<4' 'pydantic>=2.13' 'ollama>=0.6' 'fpdf2' 'huggingface_hub'

## 3. Index Chroma (bge-m3) + rutare per-thread

In [ ]:
import logging, tempfile
from pathlib import Path
logging.basicConfig(level=logging.WARNING, format='%(message)s')

# ChromaDB cache-uiește clienții per cale; rulările eșuate anterioare (în același kernel)
# lasă un client otrăvit -> "readonly database / DB moved" (code 1032). Golim cache-ul + cale nouă.
import chromadb
for _imp in ('chromadb.api.shared_system_client', 'chromadb.api.client'):
    try:
        import importlib
        importlib.import_module(_imp).SharedSystemClient.clear_system_cache(); break
    except Exception: pass

CHROMA_TMP = Path(tempfile.mkdtemp(prefix='licenta_chroma_', dir='/tmp'))
print('Chroma dir:', CHROMA_TMP)
from licenta.index import build_index
build_index(chroma_dir=CHROMA_TMP)

In [ ]:
import threading, ollama
_tls = threading.local(); _orig_chat = ollama.chat
def _routing_chat(*a, **k):
    host = getattr(_tls, 'host', None)
    return ollama.Client(host=host).chat(*a, **k) if host else _orig_chat(*a, **k)
ollama.chat = _routing_chat

def pull_and_warm(model):
    os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
    have = subprocess.run(['ollama','show',model], capture_output=True).returncode == 0
    if not have:
        print('pull', model, '...', flush=True)
        subprocess.run(['ollama', 'pull', model], check=True)
    for host in HOSTS:
        ollama.Client(host=host).chat(model=model, messages=[{'role':'user','content':'hi'}])
    print('warm OK:', model)

## 4. Helper: rulare paralelă a evaluării pentru un model
Salvează `data/eval/runs/<label>/results.json` + `report.md`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict
from tqdm.auto import tqdm
import json, time
from licenta.evaluate import load_gold_set, evaluate_case, _build_report
from licenta.retriever import Retriever

retriever = Retriever(chroma_dir=CHROMA_TMP)
cases = load_gold_set(Path('data/eval/gold_set.json'))

def run_eval(label, model, k=4, temperature=0.0, few_shot=False):
    pull_and_warm(model)
    def worker(ic):
        idx, case = ic
        _tls.host = HOSTS[idx % len(HOSTS)]
        try: return evaluate_case(case, retriever, model, k=k, temperature=temperature, few_shot=few_shot)
        except Exception as e: print('ERR', case.id, e); return None
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(HOSTS)*2) as ex:
        results = [r for r in tqdm(ex.map(worker, enumerate(cases)), total=len(cases)) if r]
    out = Path('data/eval/runs') / label; out.mkdir(parents=True, exist_ok=True)
    (out/'results.json').write_text(json.dumps([asdict(r) for r in results], ensure_ascii=False, indent=2))
    (out/'report.md').write_text(_build_report(results, model, k))
    print(f'{label}: {len(results)} cazuri in {time.time()-t0:.0f}s -> {out}')
    return results

## 5. Config — alege ce rulezi

In [ ]:
# Re-eval COMPLET pe corpusul re-scrape-uit (arbore corectat, 147 chunks). Toate 6 configurații.
# Qwen sweep + RoLlama. RoLlama3-8B (OpenLLM-Ro) e construit local din GGUF cu template Llama-3
# (vezi celula următoare); restul se pull-uiesc direct.
MODELS = [
    ('3b_q4',          'qwen2.5:3b-instruct-q4_K_M'),
    ('3b_q8',          'qwen2.5:3b-instruct-q8_0'),
    ('3b_fp16',        'qwen2.5:3b-instruct-fp16'),
    ('7b_q4',          'qwen2.5:7b-instruct-q4_K_M'),
    ('14b_q4',         'qwen2.5:14b-instruct-q4_K_M'),
    ('rollama3_8b_q4', 'rollama3ro'),
]
JUDGE_TAG = 'qwen2.5:14b-instruct-q4_K_M'   # judecător multilingv neutru, pull separat
RUN_FEWSHOT = False
RUN_R7_RECOVERY = True    # recuperarea refuzurilor spurioase prin reîncercare cu R7 activ (celula §9)
RUN_TEMP_SWEEP  = True    # mini-sweep de temperatură pe 3B-Q8 (0.0/0.3/0.7), uniformitate metodologică


In [ ]:
# Construiește RoLlama în Ollama cu template-ul Llama-3 corect (Modelfile).
# Pull-ul direct hf.co nu aplică template-ul -> modelul degenerează la "safe".
import subprocess, os
from huggingface_hub import hf_hub_download
_gguf = hf_hub_download('legraphista/RoLlama3-8b-Instruct-IMat-GGUF', 'RoLlama3-8b-Instruct.Q4_K_S.gguf')
_TPL = ('{{ if .System }}<|start_header_id|>system<|end_header_id|>\n\n'
        '{{ .System }}<|eot_id|>{{ end }}'
        '{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>\n\n'
        '{{ .Prompt }}<|eot_id|>{{ end }}'
        '<|start_header_id|>assistant<|end_header_id|>\n\n')
_Q = chr(34)*3
_mf = 'FROM ' + _gguf + '\nTEMPLATE ' + _Q + _TPL + _Q + '\nPARAMETER stop "<|eot_id|>"\n'
open('/tmp/Modelfile.ro','w').write(_mf)
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.run(['ollama','create','rollama3ro','-f','/tmp/Modelfile.ro'], check=True)
print('built rollama3ro from', _gguf)

In [ ]:
# (OPȚIONAL, recomandat) Sanity-check pe modelul RO ÎNAINTE de sweep-ul de 68 de cazuri:
# verifică (1) pull-ul hf.co, (2) că Ollama aplică template-ul Llama-3, (3) că iese JSON structurat valid.
RO_TAG = 'rollama3ro'
try:
    pull_and_warm(RO_TAG)
    from licenta.generator import generate
    _tls.host = HOSTS[0]
    q = 'Ce documente îmi trebuie pentru a înregistra nașterea copilului?'
    hits = retriever.query(q, k=4)
    a = generate(q, hits, model=RO_TAG, temperature=0.0)
    print('refusal:', a.schema.is_refusal, '| attempts:', a.attempts)
    print('answer :', a.schema.answer_text[:400])
    print('citații:', a.schema.cited_source_indices)
    print('\n-> dacă răspunsul e coerent în română și citează surse, modelul e gata de sweep.')
except Exception as e:
    print('RO sanity FAILED:', repr(e))
    print('-> încearcă fallback :Q4_K_S sau :Q8_0 în RO_TAG/MODELS, sau verifică Internet=On.')

## 6. Rulează eval pentru toate modelele

In [ ]:
all_results = {}
for label, tag in MODELS:
    try:
        all_results[label] = run_eval(label, tag)
    except Exception as e:
        print(f'SKIP {label} ({tag}): {e}')
if RUN_FEWSHOT:
    all_results['3b_q4_fewshot'] = run_eval('3b_q4_fewshot', dict(MODELS)['3b_q4'], few_shot=True)

# Mini-sweep de temperatură pe configul recomandat (3B-Q8), totul pe T4 -> sensibilitatea
# la temperatură, comparabil apples-to-apples cu restul (toate la temp 0).
if RUN_TEMP_SWEEP:
    q8 = dict(MODELS)['3b_q8']
    for T in (0.0, 0.3, 0.7):
        lbl = f'3b_q8_t{str(T).replace(".","")}'
        try:
            all_results[lbl] = run_eval(lbl, q8, temperature=T)
        except Exception as e:
            print(f'SKIP {lbl}: {e}')


## 7. Tabel comparativ (sweep + scară)
refusal accuracy, recall@4, lexical recall, contract pass, latency.

In [ ]:
import statistics as st
def summarize(results):
    n=len(results)
    ref=sum(r.refusal_correct for r in results)/n
    rr=[r for r in results if r.breadcrumb_recall is not None]
    rec=sum(1 for r in rr if r.breadcrumb_recall)/len(rr)
    kw=[r.keyword_coverage for r in results if r.keyword_coverage is not None]
    con=sum(r.contract_valid for r in results)/n
    lat=sum(r.latency_s for r in results)/n
    return dict(n=n, refusal=round(100*ref), recall=round(100*rec),
                kw=round(100*sum(kw)/len(kw)) if kw else 0, contract=round(100*con), lat=round(lat))
import pandas as pd
df = pd.DataFrame({lab: summarize(res) for lab,res in all_results.items()}).T
print(df.to_string())
df.to_csv('data/eval/runs/summary.csv')
df

## 8. LLM-judge faithfulness
Un model mare judecă, per caz non-refuz, dacă răspunsul e susținut de sursele citate:
**corect / parțial / incorect**. Metrică reală de fidelitate (vs proxy-ul lexical).

In [ ]:
import json, re

# JUDGE_TAG vine din cellula de config (pull separat, nu e în MODELS).
pull_and_warm(JUDGE_TAG)

JUDGE_SYS = ('Ești un evaluator strict. Primești o ÎNTREBARE, un RĂSPUNS și SURSELE citate. '
 'Decide dacă răspunsul este susținut DOAR de surse. Răspunde cu un singur cuvânt: '
 '"corect" (tot ce afirmă e în surse), "partial" (parțial susținut sau incomplet) sau '
 '"incorect" (afirmă lucruri absente din surse sau greșite).')

def judge_one(case_result):
    if case_result['is_refusal'] or not case_result.get('cited_sources_text'):
        return None
    srcs = '\n'.join(f'[S{i+1}] {t}' for i,t in enumerate(case_result['cited_sources_text']))
    msg = f"ÎNTREBARE: {case_result['question']}\n\nRĂSPUNS: {case_result['answer_text']}\n\nSURSE:\n{srcs}"
    _tls.host = HOSTS[0]
    r = ollama.chat(model=JUDGE_TAG, messages=[{'role':'system','content':JUDGE_SYS},{'role':'user','content':msg}],
                    options={'temperature':0})
    w = r['message']['content'].strip().lower()
    return 'corect' if 'corect' in w else ('partial' if 'partial' in w or 'parțial' in w else 'incorect')

def judge_model(label):
    rows = json.loads(Path(f'data/eval/runs/{label}/results.json').read_text())
    verdicts = [v for v in (judge_one(r) for r in rows) if v]
    from collections import Counter; c = Counter(verdicts)
    n = len(verdicts) or 1
    print(f'{label}: judge n={len(verdicts)} | corect {100*c["corect"]//n}% partial {100*c["partial"]//n}% incorect {100*c["incorect"]//n}%')
    Path(f'data/eval/runs/{label}/judge.json').write_text(json.dumps(dict(c), ensure_ascii=False))
    return c

for label in all_results:
    try: judge_model(label)
    except Exception as e: print('judge skip', label, e)

## 9. R7 recovery (refuz spurious recuperat)
Rulează cazurile spurioase + control prin bucla cu R7 activ; câte refuzuri spurioase devin răspunsuri,
fără a sparge refuzurile corecte.

In [ ]:
if RUN_R7_RECOVERY:
    base_label = '3b_q4'
    base = json.loads(Path(f'data/eval/runs/{base_label}/results.json').read_text())
    spur = [r['case_id'] for r in base if r['is_refusal'] and not r['should_refuse']]
    ctrl = [r['case_id'] for r in base if r['should_refuse']]
    gold = {c.id: c for c in cases}
    from licenta.generator import generate
    tag = dict(MODELS)['3b_q4']; pull_and_warm(tag)
    rows=[]
    def w(ic):
        idx,cid = ic; _tls.host = HOSTS[idx % len(HOSTS)]
        case = gold[cid]; hits = retriever.query(case.question, k=4)
        a = generate(case.question, hits, model=tag, temperature=0.0)
        return dict(id=cid, subset=('control' if case.should_refuse else 'spurious'),
                    base_refused=(cid in spur or cid in [r['case_id'] for r in base if r['is_refusal']]),
                    final_refused=a.schema.is_refusal, attempts=a.attempts)
    with ThreadPoolExecutor(max_workers=len(HOSTS)*2) as ex:
        rows = list(tqdm(ex.map(w, enumerate(spur+ctrl)), total=len(spur+ctrl)))
    sp=[r for r in rows if r['subset']=='spurious']; ct=[r for r in rows if r['subset']=='control']
    rec=sum(1 for r in sp if not r['final_refused'])
    kept=sum(1 for r in ct if r['final_refused'])
    print(f'R7 recovery: spurious recuperate {rec}/{len(sp)} | control pastrate {kept}/{len(ct)}')
    Path('data/eval/runs/r7_recovery.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2))

In [ ]:
# Analize de mecanism pe baseline 3b_q4 (temp 0, n=98) — rezultate auditabile, intră în bundle.
# Rulează ca subprocese -> host unic 11434, indexul Chroma din CHROMA_TMP. num_predict cap activ.
_env = f'LICENTA_CHROMA_DIR={CHROMA_TMP} OLLAMA_HOST=127.0.0.1:11434'
# scor top-1 retrieval pe tot gold-ul (pt distribuția R7 + calibrarea pragului)
!{_env} python scripts/retrieval_scores.py data/eval/runs/3b_q4
# ablație contracte ON/OFF (first_attempt vs post-retry) -> data/eval/ablation_contracts.json
!{_env} python scripts/ablation_contracts.py
# recuperare refuz spurios cu prompt reformulat -> data/eval/runs/refusal_experiment_*.json
!{_env} python scripts/refusal_experiment.py data/eval/runs/3b_q4
print('Analize de mecanism: gata.')

## 10. Bundle rezultatele (descarcă results.zip)

In [ ]:
import shutil
# 'data/eval' (nu doar runs/) ca să prindă și ablation_contracts.json + retrieval_scores.json
shutil.make_archive('/kaggle/working/results', 'zip', 'data/eval')
print('Gata -> /kaggle/working/results.zip')
!ls -la /kaggle/working/results.zip